---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-33: LangChain <b>Chains</b> Component</h1>

# Learning agenda of this notebook

> **The idea of this lecture:** real tasks need more than one step: fill a prompt, call a model, clean the answer, then maybe call another model with that answer. A **chain** joins these steps with the `|` operator, so that the output of one step becomes the input of the next, and the whole pipeline runs with a single `invoke()`. This way of writing chains is called **LCEL**, the LangChain Expression Language.

1. **Where Chains Fit**
    - What a chain is, and what happened to the old chain classes
    - Setup: check the API keys
2. **Your First Chain**
    - Example 1: `prompt | model | parser`, compared with the same three steps by hand
    - Example 2: A chain is a Runnable too: `batch()` and `stream()`
3. **Output Parsers**
    - Example 3: `PydanticOutputParser`: from the model's text to a Python object
4. **Three Shapes of Chains** (each starts with a warm-up without a model)
    - Example 4: Sequential: translate, then summarise (`RunnableLambda`)
    - Example 5: Parallel: two reports at the same time, then one merged report (`RunnableParallel`)
    - Example 6: Branching (if-else): a different reply for positive, negative and mixed feedback (`RunnableBranch`)
5. **Where Chains Go Next**
6. **Summary**

> **Note:** all examples use the free model `openai/gpt-oss-120b` on Groq.

# <span style='background :lightgreen' >1. Where Chains Fit</span>

In the app of Lec-30, a **chain** is what connects the building blocks: retriever, prompt template and model, one after the other.

<div style="text-align:center;">
    <img src="../images/langchain-new-03.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## What is a chain?
- In Lec-31 we filled a template with `invoke()` and gave the result to a model with another `invoke()`. Prompt templates, models and output parsers all have the **same** `invoke()` method: in LangChain they are all **Runnables**.
- Because they share one interface, they can be joined with `|`: `prompt | model | parser` is a new Runnable that calls the three in order and passes each output to the next. This is a **chain**.
- A chain is a Runnable again, so it has the same methods (`invoke`, `batch`, `stream`, …) and can itself be a step of a bigger chain.

## What happened to the old chain classes
Older tutorials build chains with classes. They still work, but they were moved to `langchain-classic` in LangChain 1.0 and print a deprecation warning:

| Old class (`langchain_classic.chains`) | Today, with LCEL (`langchain_core`) |
|---|---|
| `LLMChain(llm=model, prompt=prompt)` | `prompt \| model \| StrOutputParser()` (Example 1) |
| `SimpleSequentialChain`, `SequentialChain` | `chain_1 \| reshape \| chain_2` (Example 4) |
| `RouterChain`, `MultiPromptChain` | `RunnableBranch(...)` (Example 6) |

For workflows with **loops**, saved **state** or several **agents**, LangChain now recommends **LangGraph**, which we meet after the agent lectures. Chains are for fixed pipelines that run from start to end.

> **Documentation:** https://reference.langchain.com/python/langchain-core/runnables

## Setup: check the API keys
- Every example below is **standalone**: its first code cell loads the API keys and creates the model with `init_chat_model("groq:openai/gpt-oss-120b")` (Lec-30). The later cells of the same example build on it, so run them in order.
- To use OpenAI instead (paid), replace `"groq:openai/gpt-oss-120b"` with `"openai:gpt-4o-mini"` in an example's first cell.
- This cell only checks that the Groq key is found; no example depends on it.

In [1]:
import os                                                   # read the environment variables
from dotenv import load_dotenv                              # load the API keys from ../keys/.env

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ (the file wins)

if os.getenv("GROQ_API_KEY"):                               # the only key the examples need (Groq, in every example)
    print("GROQ_API_KEY set")
else:
    print("GROQ_API_KEY MISSING")                           # never print any part of a key

GROQ_API_KEY set


# <span style='background :lightgreen' >2. Your First Chain</span>

<div style="text-align:center;">
    <img src="../images/langchain-new-16.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 1a: <code>prompt | model | parser</code>, compared with the same three steps by hand</span>
### Three steps by hand
We translate a short biography into Urdu. Each step's output is passed **by our code** to the next step.

In [5]:
# Step 1: create the prompt template
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders (Lec-31)

# {language} and {text} are placeholders: invoke() fills them from a dict with the same two keys
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")
print("Placeholders to fill:", translate_prompt.input_variables)   # ['language', 'text']

Placeholders to fill: ['language', 'text']


In [6]:
# Step 2: fill the template → a PromptValue

# the English text we need to translate ('''...''' lets a string run over several lines)
BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

# the two place holders one value for each {placeholder}: the keys must match the names
prompt_value = translate_prompt.invoke({"language": "Urdu", "text": BIO} )        
print(prompt_value)
print()
print()

# PromptValue → a plain str, only to look at it in a human readable way
prompt_text = prompt_value.to_string()                      
print(prompt_text)

text='Translate the following English text into Urdu. Reply with the translation only.\n\nDr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.\nHe holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching\nand management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a\ntechnology entrepreneur and the founder of Excaliat and Falcon-Hunt.'


Translate the following English text into Urdu. Reply with the translation only.

Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrep

In [7]:
# Step 3: load the API keys, create the model, and invoke the model with prompt_value

from dotenv import load_dotenv                              # reads the API keys from a file
from langchain.chat_models import init_chat_model           # creates a chat model from "provider:model"

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",         # free, on Groq. Yiu can use  "openai:gpt-4o-mini" which is paid on OpenAI
                        max_retries=5,                      # wait and retry when the free rate limit is hit
                        )

# call the model → an AIMessage
ai_message = model.invoke(prompt_value)                     # PromptValue → AIMessage (one call to Groq)

# An AIMessage holds more than the reply text like usage_metadata (the token counts) and response_metadata (the model name, why it stopped). Most of the time we only want the text.
print(ai_message)

content='ڈاکٹر محمد عارف بٹ ڈیپارٹمنٹ آف ڈیٹا سائنس، یونیورسٹی آف دی پنجاب، لاہور میں اسسٹنٹ پروفیسر ہیں۔ ان کے پاس ایم ایس سی، ایم فل (دونوں گولڈ میڈل کے ساتھ) اور کمپیوٹر سائنس میں پی ایچ ڈی ہے۔ 33 سے زائد سال کی تدریس اور انتظامیہ کے تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سکیورٹی اور مصنوعی ذہانت پڑھاتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایکسیلیئٹ اور فیلکن ہنٹ کے بانی ہیں۔' additional_kwargs={'reasoning_content': 'We need to translate English text into Urdu, reply with translation only. No extra text. Provide Urdu translation. Ensure proper Urdu script. Let\'s translate.\n\n"Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.\nHe holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a technology entrepr

In [8]:
# Step 4: keep only the text → a str
translation = ai_message.text                               # AIMessage → str: here OUR code does this step, by hand
print(translation)

ڈاکٹر محمد عارف بٹ ڈیپارٹمنٹ آف ڈیٹا سائنس، یونیورسٹی آف دی پنجاب، لاہور میں اسسٹنٹ پروفیسر ہیں۔ ان کے پاس ایم ایس سی، ایم فل (دونوں گولڈ میڈل کے ساتھ) اور کمپیوٹر سائنس میں پی ایچ ڈی ہے۔ 33 سے زائد سال کی تدریس اور انتظامیہ کے تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سکیورٹی اور مصنوعی ذہانت پڑھاتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایکسیلیئٹ اور فیلکن ہنٹ کے بانی ہیں۔


## <span style='color:#7e22ce'>Example 1b: <code>prompt | model | parser</code>, creating a chain and invoking it</span>
- **`StrOutputParser()`** is the simplest output parser: it takes the `AIMessage` from the model and returns only its text, a `str`, the same as `.text` in Step 4 of Example 1a. The metadata (token counts, ids) is dropped, and a plain `str` goes through unchanged.
- Because it is a component, it can be the last step of a chain: `translate_prompt | model | parser`. One `invoke()` runs all three steps.

In [1]:
# Creating  a chain and invoking it (prompt | model | parser)

from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the prompt template, with {language} and {text} placeholders
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")

# Create the chat model 
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",
                        max_retries=5                                     # wait and retry when the free rate limit is hit
                        )

# You can create two chains
chain1 = translate_prompt | model                      # returns an AIMessage
chain2 = translate_prompt | model | StrOutputParser()  # StrOutputParser() is the simplest output parser that takes the AIMessage from the model and returns only its text


In [2]:
# Invoke the whole chain once by passing it the required place holders for the prompt template and print the translation

# The English text to translate
BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

# Invoke chain1, to extract the text  from the response and drop the metadata (token counts, ids), you need to use .text
translation1 = chain1.invoke({"language": "Urdu", "text": BIO})
print(translation1.text)

print()
print()

# chain2 is using StrOutputParser() at the end, so it automatically extracts the text and drop the metadata, and you DONOT need to use .text
translation2 = chain2.invoke({"language": "Urdu", "text": BIO})
print(translation2)

ڈاکٹر محمد عارف بٹ یونیورسٹی آف دی پنجاب، لاہور کے ڈیٹا سائنس ڈیپارٹمنٹ میں اسسٹنٹ پروفیسر ہیں۔ ان کے پاس ایم ایس سی، ایم فل (دونوں گولڈ میڈل کے ساتھ) اور کمپیوٹر سائنس میں پی ایچ ڈی ہے۔ تدریس اور مینجمنٹ میں 33 سے زائد سال کے تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سکیورٹی اور مصنوعی ذہانت کی تعلیم دیتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایکسکالیٹ اور فالکن‑ہنٹ کے بانی ہیں۔


ڈاکٹر محمد عارف بٹ ڈیٹا سائنس کے شعبہ میں اسسٹنٹ پروفیسر ہیں، یونیورسٹی آف دی پنجاب، لاہور میں۔ ان کے پاس ایم ایس سی، ایم فل (دونوں گولڈ میڈل کے ساتھ) اور کمپیوٹر سائنس میں پی ایچ ڈی ہے۔ 33 سے زیادہ سال کی تدریسی اور انتظامی تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور مصنوعی ذہانت کی تعلیم دیتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایکسیلیئٹ اور فالن‑ہنٹ کے بانی ہیں۔


In [4]:
# drawing of the chain: the input, each component, and the output
graph = chain2.get_graph()                         # every step of the chain becomes one box
graph.print_ascii()                                # the stars (*) show where each output goes; needs the grandalf package

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


## <span style='color:#7e22ce'>Example 2: A chain is a Runnable so you can call <code>stream()</code> and <code>batch()</code></span>
#### `stream(...)` yields the output piece by piece. Because the last step is `StrOutputParser`, the pieces are plain strings.

In [5]:
from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the prompt template, with {language} and {text} placeholders
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")

# Create the chat model 
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",
                        max_retries=5                                     # wait and retry when the free rate limit is hit
                        )

# Create the chain
translate_chain = translate_prompt | model | StrOutputParser()   # dict → PromptValue → AIMessage → str

# The English text to translate
BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

# The model sends small AIMessageChunk pieces; StrOutputParser passes on the text of each piece at once,
# so every piece we get here is a short str
for piece in translate_chain.stream({"language": "Urdu", "text": BIO}):                # each piece is a str (thanks to StrOutputParser)
    print(piece, end="", flush=True)                        # end="": no new line; flush=True: show it at once


ڈاکٹر محمد عارف بٹ یونیورسٹی آف دی پنجاب، لاہور کے ڈیٹا سائنس ڈیپارٹمنٹ میں اسسٹنٹ پروفیسر ہیں۔ ان کے پاس ایم ایس سی، ایم فل (دونوں سونے کے میڈل کے ساتھ) اور کمپیوٹر سائنس میں پی ایچ ڈی ہے۔ 33 سال سے زائد تدریس اور انتظامیہ کے تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور مصنوعی ذہانت پڑھاتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایکسیلیئٹ اور فالکن‑ہنٹ کے بانی ہیں۔

### `batch([...])` runs the chain on several inputs at the same time and returns the outputs in the same order.

In [1]:
from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the prompt template, with {language} and {text} placeholders
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")

# Create the chat model 
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",
                        max_retries=5                                     # wait and retry when the free rate limit is hit
                        )

# Create the chain
translate_chain = translate_prompt | model | StrOutputParser()   # dict → PromptValue → AIMessage → str

# We need to translate the message in three languages
message = "Knowledge is the light that guides us through the darkest times."
languages = ["Urdu", "French", "Arabic"]

# Create a list of three dictionaries to be passed to the .batch()
inputs = []                                                 # one input dict per language
for language in languages:
    inputs.append({"language": language, "text": message})

print(inputs) # printing for your understanding

[{'language': 'Urdu', 'text': 'Knowledge is the light that guides us through the darkest times.'}, {'language': 'French', 'text': 'Knowledge is the light that guides us through the darkest times.'}, {'language': 'Arabic', 'text': 'Knowledge is the light that guides us through the darkest times.'}]


In [2]:
# batch() runs the whole chain once per input dict, the runs at the same time (in threads).

outputs = translate_chain.batch(inputs)             # a list of str, in the same order as inputs
outputs

['علم وہ روشنی ہے جو ہمیں سب سے تاریک لمحات میں رہنمائی کرتی ہے۔',
 'La connaissance est la lumière qui nous guide à travers les temps les plus sombres.',
 'المعرفة هي النور الذي يوجهنا عبر أحلك الأوقات.']

# <span style='background :lightgreen' >3. Output Parsers</span>

An **output parser** is the last step of many chains: it turns the model's answer into the type our program needs.

| Parser | Turns the answer into | How |
|---|---|---|
| `StrOutputParser()` | a `str` | takes the text of the `AIMessage` |
| `PydanticOutputParser(pydantic_object=Book)` | a validated `Book` object | asks for JSON in the prompt, then reads and checks it |


A model can only send back **text**. To get a Python object with checked fields, `PydanticOutputParser` works in two halves, around the model:

| Method | When | What it does |
|---|---|---|
| `get_format_instructions()` | **before** the model, once | builds, from our Pydantic class, a text that tells the model which JSON to reply with. We put it into the prompt with `partial()` (Lec-31, Example 3). |
| `parse(text)` | **after** the model | reads the JSON in a string (removing any markdown code fences around it), checks every field and its type, and returns a `Book` object. If the JSON is wrong, it raises an `OutputParserException` instead of letting bad data into the program. |
| `invoke(ai_message)` | what the chain calls | takes the text of the `AIMessage` and calls `parse()` on it |

## <span style='color:#7e22ce'>Example 3: <code>PydanticOutputParser</code>: from the model's text to a Python object</span>

In [1]:
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
import rich

load_dotenv("../keys/.env", override=True)                  # load the API key
model = init_chat_model("groq:openai/gpt-oss-120b", temperature=0)         # a free model on Groq

# 1. The answer we want, as a Pydantic class: one field per value we ASK for
#    - the TYPE (str, int) is what the parser checks later: "year" must be a whole number
#    - the DESCRIPTION is copied into the format instructions, so the model reads it
class Book(BaseModel):  # the object we want back
    title: str = Field(description="the book's full title")             # must be a string
    author: str = Field(description="the author's full name")           # must be a string
    year: int = Field(description="the year the book was published")         # must be an integer
    publisher: str = Field(description="the publisher's name")                # must be a string
    isbn: str = Field(description="the book's 13-digit ISBN, digits only")    # a string, not an int: an ISBN is an identifier, not a number to calculate with

# 2. Make a parser for that class
parser = PydanticOutputParser(pydantic_object=Book)

# 3. The parser writes instructions that tell the model which JSON to reply with
# The printed text has three parts:
#   1. "The output should be formatted as a JSON instance ..."  → reply with JSON
#   2. the "foo" example                                        → one right and one wrong JSON, so the model sends
#                                                                 the VALUES, not a copy of the schema
#   3. "Here is the output schema:"                             → our Book class as a JSON schema: each field
#                                                                 with its type and description, and "required" (must be there)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "the book's full title", "title": "Title", "type": "string"}, "author": {"description": "the author's full name", "title": "Author", "type": "string"}, "year": {"description": "the year the book was published", "title": "Year", "type": "integer"}, "publisher": {"description": "the publisher's name", "title": "Publisher", "type": "string"}, "isbn": {"description": "the book's 13-digit ISBN, digits only", "title": "Isbn", "type": "string"}}, "required": ["title", "author", "year", "publisher", "isbn"]}
``

In [2]:
# 4. Put those instructions into the prompt
prompt = PromptTemplate.from_template("Give the publication year, publisher and ISBN of the book '{title}' by {author}.\n{format_instructions}")
prompt = prompt.partial(format_instructions=parser.get_format_instructions())
print(prompt)

input_variables=['author', 'title'] input_types={} partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"title": {"description": "the book\'s full title", "title": "Title", "type": "string"}, "author": {"description": "the author\'s full name", "title": "Author", "type": "string"}, "year": {"description": "the year the book was published", "title": "Year", "type": "integer"}, "publisher": {"description": "the publisher\'s name", "title": "Publisher", "type": "string"}, "isbn": {"description": "the book\'s 13-digit ISBN, digits onl

In [3]:
# 5. The chain: prompt → model → parser (the parser turns the model's JSON text into a Book object)
chain = prompt | model | parser

book = chain.invoke({"title": "Generative Deep Learning (2nd Edition)", "author": "David Foster"})

# 6. The result is a Book object, so we read each value with a dot
print(book)

title='Generative Deep Learning (2nd Edition)' author='David Foster' year=2022 publisher="O'Reilly Media" isbn='9781098103515'


> **`PydanticOutputParser` or `with_structured_output()`?** Both give a validated object. `with_structured_output()` (Lec-30, Example 9) uses the model provider's built-in JSON mode or tool calling, so it is more reliable when the model supports it. `PydanticOutputParser` only needs text, so it works with **any** model, including small local ones, and many RAG tutorials use it.

# <span style='background :lightgreen' >4. Three Shapes of Chains</span>

<div style="text-align:center;">
    <img src="../images/langchain-new-17.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 4: Sequential: translate, then summarise (<code>RunnableLambda</code>)</span>
- **`RunnableLambda(function)`** wraps an ordinary Python function so that it becomes a Runnable: a chain step with `invoke()`, `batch()` and `|`.
- **The rules:** the function takes exactly **one** argument (the output of the step before it); what it returns is the input of the step after it; to pass several values onward, return a **dict**.
- **Why we need it here:** `translate_chain` returns a **str**, but `summarize_prompt` needs a **dict** with `language` and `text`. A `RunnableLambda` sits between them as an **adapter**.
- The first cell is a warm-up without a model; the cells after it use `RunnableLambda` in the chain.

In [1]:
# Warm-up (no model, no tokens): RunnableLambda turns an ordinary Python function into a chain step
from langchain_core.runnables import RunnableLambda             # wraps a Python function → a Runnable

# Step 1: an ordinary Python function, called the normal way
def count_words(text):
    '''str → int: how many words are in the text.'''
    words = text.split()                                        # split at the spaces → a list of words
    count = len(words)                                          # how many items the list has
    return count                                                # an int

result = count_words("Knowledge is the light")                  # a normal function call, nothing new yet
print("Step 1, plain function:", result)                        # 4

# Step 2: wrap the SAME function in RunnableLambda → now it has invoke(), batch() and |
count_step = RunnableLambda(count_words)                        # pass the function itself: no () after its name
result = count_step.invoke("Knowledge is the light")            # invoke() calls count_words("Knowledge is the light") for us
print("Step 2, invoke():      ", result)                        # 4: the same answer as Step 1

results = count_step.batch(["one two", "one two three"])        # batch() calls count_words once for each item of the list
print("Step 2, batch():       ", results)                       # [2, 3]: one answer per input, in the same order

# Step 3: the function takes ONE argument; to send several values onward, it returns a dict
def make_report(count):
    '''int → dict. Its ONE argument is whatever the step before it returned.'''
    report = {}                                                 # start with an empty dict
    report["words"] = count                                     # the number that came from the step before
    report["long"] = count > 5                                  # True when the text has more than 5 words
    return report                                               # this dict is the input of the next step

report_step = RunnableLambda(make_report)                       # wrap this function too
result = report_step.invoke(4)                                  # test it alone with a number
print("Step 3, alone:         ", result)                        # {'words': 4, 'long': False}

# Step 4: join the two steps with |: the output of count_step becomes the input of report_step
word_chain = count_step | report_step                           # str ──count_step──► int ──report_step──► dict
result = word_chain.invoke("Knowledge is the light that guides us")   # 7 words go in as ONE str
print("Step 4, the chain:     ", result)                        # {'words': 7, 'long': True}

Step 1, plain function: 4
Step 2, invoke():       4
Step 2, batch():        [2, 3]
Step 3, alone:          {'words': 4, 'long': False}
Step 4, the chain:      {'words': 7, 'long': True}


In [2]:
# Step 1: load the API keys, create the model, and create the two chains

from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain_core.runnables import RunnableLambda             # turns a Python function into a chain step
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the chat model
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",             # free, on Groq
                        max_retries=5,                          # wait and retry when the free rate limit is hit
                        )

# Chain 1, the same as in Example 1:   {language, text} → PromptValue → AIMessage → str (the translation)
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")
translate_chain = translate_prompt | model | StrOutputParser()

# Chain 2, new:                        {language, text} → PromptValue → AIMessage → str (the summary)
summarize_prompt = PromptTemplate.from_template("Summarise the following {language} text in fewer than 50 words, in {language}:\n\n{text}")
summarize_chain = summarize_prompt | model | StrOutputParser()

# The problem: Chain 1 gives back a str, but Chain 2 needs a dict with these two keys
print("summarize_prompt needs:", summarize_prompt.input_variables)   # ['language', 'text']

summarize_prompt needs: ['language', 'text']


In [3]:
# Step 2: the adapter: a RunnableLambda that turns the str of Chain 1 into the dict that Chain 2 needs

def make_summary_input(urdu_text):
    '''str → dict. urdu_text is the ONE argument: whatever translate_chain returned (the Urdu str).'''
    summary_input = {}                                          # start with an empty dict
    summary_input["language"] = "Urdu"                          # fills {language} of summarize_prompt
    summary_input["text"] = urdu_text                           # fills {text} of summarize_prompt
    return summary_input                                        # this dict is what summarize_chain receives next

to_summary_input = RunnableLambda(make_summary_input)           # the function as a chain step, as in the warm-up

# Test the adapter alone before putting it into a chain: no model, no tokens
test = to_summary_input.invoke("یہ ایک مثال ہے۔")               # a short Urdu sentence: "This is an example."
print(test)                                                     # {'language': 'Urdu', 'text': 'یہ ایک مثال ہے۔'}

{'language': 'Urdu', 'text': 'یہ ایک مثال ہے۔'}


In [4]:
# Step 3: join the three steps into one chain and invoke it once

# dict ──translate_chain──► str (Urdu) ──to_summary_input──► dict ──summarize_chain──► str (the summary)
translate_then_summarize = translate_chain | to_summary_input | summarize_chain

# The English text to translate (the same as in Example 1)
BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

# The input of the FIRST step: translate_prompt needs {language} and {text}
values = {"language": "Urdu", "text": BIO}

# One invoke() runs everything: English → Urdu → a short Urdu summary (two model calls)
summary = translate_then_summarize.invoke(values)
print(summary)

ڈاکٹر محمد عارف بٹ، یونیورسٹی آف پنجاب کے اسسٹنٹ پروفیسر، ایم ایس سی، ایم فل (سونے کے میڈل) اور پی ایچ ڈی کے حامل، 33 سال سے زیادہ تدریسی و انتظامی تجربے کے ساتھ، سسٹم پروگرامنگ، سائبر سیکیورٹی اور AI پڑھاتے ہیں، اور ایکسیلیئٹ و فیلکن‑ہنٹ کے بانی ہیں۔


In [5]:
# Step 4: drawing of the chain: make_summary_input is a box of its own, between the two chains
graph = translate_then_summarize.get_graph()                    # every step of the chain becomes one box
graph.print_ascii()                                             # the stars (*) show where each output goes

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
  +--------------------+   
  | make_summary_input |   
  +--------------------+   
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

## <span style='color:#7e22ce'>Example 5: Parallel: two reports at the same time, then one merged report (<code>RunnableParallel</code>)</span>
- **`RunnableParallel({"key_a": step_a, "key_b": step_b})`** gives the **same** input to every step in the dict, runs them **at the same time**, and returns **one dict**: our keys, each holding what its step returned.
- That dict can go straight into the next prompt, if its placeholders have the **same names** as the keys:
```
{topic} ─┬─► chain_american       ─► str ─► "report_a" ─┐
         └─► chain_middle_eastern ─► str ─► "report_b" ─┴─► {report_a, report_b} ─► merge_chain ─► str
```
- On Groq's free tier (about 8,000 tokens per minute) the second report may have to wait for the first, so the time saved varies. The warm-up shows the saving without a model.

In [1]:
# Warm-up (no model, no tokens): RunnableParallel gives ONE input to several steps, at the same time
import time                                                     # sleep(), and a clock to measure the time
from langchain_core.runnables import RunnableLambda             # wraps a Python function → a Runnable
from langchain_core.runnables import RunnableParallel           # runs several Runnables on the same input, at the same time

# Step 1: two ordinary functions; each waits 1 second, like a slow model
def slow_upper(text):
    '''str → str: the text in capital letters, after 1 second.'''
    time.sleep(1)                                               # wait 1 second
    upper = text.upper()                                        # "chains" → "CHAINS"
    return upper

def slow_length(text):
    '''str → int: the number of characters, after 1 second.'''
    time.sleep(1)                                               # wait 1 second
    length = len(text)                                          # "chains" → 6
    return length

# Step 2: RunnableParallel takes a dict: key → the step that fills it (we choose the key names)
both_steps = RunnableParallel({
    "upper": RunnableLambda(slow_upper),                        # the key "upper" will hold what slow_upper returns
    "length": RunnableLambda(slow_length),                      # the key "length" will hold what slow_length returns
})

start = time.perf_counter()                                     # read the clock before
result = both_steps.invoke("chains")                            # "chains" goes to BOTH functions; they run at the same time
end = time.perf_counter()                                       # read the clock after
seconds = end - start                                           # how long invoke() took

print("Step 2, the result:", result)                            # ONE dict with OUR keys: {'upper': 'CHAINS', 'length': 6}
print(f"Step 2, two steps of 1 second each took {seconds:.1f} seconds")   # about 1, not 2: they ran at the same time

# Step 3: the next step receives that WHOLE dict and reads its keys (merge_chain will do the same)
def combine(data):
    '''dict → str. Its ONE argument is the dict that RunnableParallel returned.'''
    upper = data["upper"]                                       # what slow_upper returned
    length = data["length"]                                     # what slow_length returned
    sentence = f"{upper} has {length} letters"
    return sentence

combine_step = RunnableLambda(combine)                          # wrap it, as in Example 4
parallel_chain = both_steps | combine_step                      # str ──both_steps──► {upper, length} ──combine_step──► str
result = parallel_chain.invoke("chains")
print("Step 3, the chain: ", result)                            # CHAINS has 6 letters

Step 2, the result: {'upper': 'CHAINS', 'length': 6}
Step 2, two steps of 1 second each took 1.0 seconds
Step 3, the chain:  CHAINS has 6 letters


In [2]:
# Step 1: load the API keys, create the model, and create one report chain per perspective

from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain_core.runnables import RunnableParallel           # runs several chains on the same input, at the same time
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the chat model
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",             # free, on Groq
                        max_retries=5,                          # wait and retry when the free rate limit is hit
                        )

# One report template with two placeholders: {topic} and {perspective}
# "in about 300 words" keeps both reports short enough for the merge step: Groq's free tier rejects
# any single request above 8,000 tokens (HTTP 413), and two long reports together can exceed that
report_prompt = PromptTemplate.from_template("Write a detailed analytical report, in about 300 words, about the {topic} from {perspective}.")

# Two versions of the same template: partial() (Lec-31) fixes {perspective}, so each chain needs only {topic}
american_prompt = report_prompt.partial(perspective="the American perspective")
chain_american = american_prompt | model | StrOutputParser()               # {topic} → PromptValue → AIMessage → str (a report)

middle_eastern_prompt = report_prompt.partial(perspective="the Middle Eastern perspective")
chain_middle_eastern = middle_eastern_prompt | model | StrOutputParser()   # {topic} → PromptValue → AIMessage → str (a report)

print("american_prompt needs:      ", american_prompt.input_variables)         # ['topic']: {perspective} is already filled
print("middle_eastern_prompt needs:", middle_eastern_prompt.input_variables)   # ['topic']

american_prompt needs:       ['topic']
middle_eastern_prompt needs: ['topic']


In [3]:
# Step 2: RunnableParallel gives the SAME input to both chains, and returns ONE dict
#   {topic} ─┬─► chain_american       ─► str ─► "report_a"
#            └─► chain_middle_eastern ─► str ─► "report_b"
both_reports = RunnableParallel({
    "report_a": chain_american,                                 # key → the chain that fills it (we choose the key names)
    "report_b": chain_middle_eastern,                           # the two chains run in two threads, at the same time
})

# ONE input: both chains get this same dict
topic = {"topic": "9/11 incident"}

# invoke() returns when BOTH chains are done (two model calls)
reports = both_reports.invoke(topic)
print(type(reports).__name__, "with the keys", list(reports.keys()))   # dict with the keys ['report_a', 'report_b']
print()

# Print the first 200 characters of each report
for key in reports:
    report = reports[key]                                       # the whole report stored under this key (a str)
    print(f"{key}: {report[:200]} …")
    print()

dict with the keys ['report_a', 'report_b']

report_a: **Analytical Report – The September 11, 2001 Terrorist Attacks (American Perspective)**  

**Executive Summary**  
On September 11, 2001, four coordinated hijackings by al‑Qaeda operatives resulted in …

report_b: **Analytical Report – The 9/11 Attacks as Viewed from the Middle East (≈300 words)**  

**1. Immediate Regional Reaction**  
Within hours of the September 11, 2001 attacks, governments across the Midd …



In [4]:
# Step 3: the merge chain: its placeholders {report_a} and {report_b} have the SAME names as the keys of Step 2
# (if a key and a placeholder do not match, invoke() raises a KeyError)
merge_prompt = PromptTemplate.from_template("Combine the following two reports into one unified report of fewer than 50 words.\nReport A: {report_a}\nReport B: {report_b}")
merge_chain = merge_prompt | model | StrOutputParser()          # {report_a, report_b} → PromptValue → AIMessage → str

# The dict of Step 2 goes straight in (one model call)
merged = merge_chain.invoke(reports)
print(merged)

On 9/11 al‑Qaeda hijacked four planes, killing 2,977, prompting U.S. security overhaul, the PATRIOT Act, wars in Afghanistan/Iraq, and intelligence reforms. Middle‑East governments expressed condolence, deepened U.S. security ties, cracked down on radicals, and debated U.S. policies, reshaping regional geopolitics.


In [5]:
# Step 4: the whole pipeline as ONE chain: {topic} → {report_a, report_b} → one short report
final_chain = both_reports | merge_chain                        # a RunnableParallel is a Runnable too, so | works with it

# Two reports at the same time, then the merge (three model calls)
final_report = final_chain.invoke(topic)
print(final_report)

On 9/11 al‑Qaida hijacked four planes, killing 2,977, prompting U.S. security overhauls (PATRIOT Act, DHS) and wars in Afghanistan and Iraq. Middle‑Eastern states condemned the attacks, tightened U.S. ties, suppressed dissent, while fringe jihadists praised them, shaping regional politics and anti‑U.S. sentiment.


In [6]:
# Step 5: drawing of the chain: the two report chains side by side, joining again before the merge
graph = final_chain.get_graph()                                 # every step of the chain becomes one box
graph.print_ascii()                                             # the stars (*) show where each output goes

       +----------------------------------+        
       | Parallel<report_a,report_b>Input |        
       +----------------------------------+        
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
    +----------+                  +----------+     
    | ChatGroq |                  | ChatGroq |     
    +----------+                  +----------+     
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP

## <span style='color:#7e22ce'>Example 6: Branching (if-else): a different reply for positive, negative and mixed feedback (<code>RunnableBranch</code>)</span>
- **`RunnablePassthrough.assign(key=step)`** passes the input dict through **unchanged** and **adds** one key holding what `step` returned: `{"feedback": …}` becomes `{"feedback": …, "sentiment": "mixed"}`. Without it, the next step would get only the label and the customer's words would be lost.
- **`RunnableBranch((condition, step), (condition, step), default_step)`** is an `if` / `elif` / `else` for chains. Each condition is a function that gets the **whole dict** and returns `True` or `False`. The conditions are checked in order, only the step of the **first** true one runs, and the last argument (no condition) runs when none is true.
- The two warm-ups teach these two functions on numbers (positive, negative, zero), without a model. The classifier then uses `with_structured_output()` (Lec-30, Example 9) with `Literal[...]`, so its label is always one of the three words.

In [1]:
# Warm-up 1 (no model, no tokens): RunnablePassthrough.assign() keeps the input dict and ADDS one key
from langchain_core.runnables import RunnableLambda             # wraps a Python function → a Runnable
from langchain_core.runnables import RunnablePassthrough        # passes its input through unchanged

# Step 1: a classifier function: it gets the whole dict and returns one label (a str)
def find_sign(data):
    '''{"number": -3} → "negative". Its ONE argument is the whole input dict.'''
    number = data["number"]                                     # read the number from the dict
    if number > 0:
        return "positive"
    elif number < 0:
        return "negative"
    else:
        return "zero"

sign_step = RunnableLambda(find_sign)                           # wrap it, as in Example 4
result = sign_step.invoke({"number": -3})
print("Step 1, the step alone: ", result)                       # 'negative': only the label, the number is LOST

# Step 2: .assign(sign=sign_step) runs sign_step on the dict, and stores what it returns under the key "sign"
#         The argument name (sign=) becomes the name of the new key.
#         In plain Python it does:   new = dict(old)   then   new["sign"] = sign_step.invoke(old)
add_sign = RunnablePassthrough.assign(sign=sign_step)
result = add_sign.invoke({"number": -3})
print("Step 2, with .assign(): ", result)                       # {'number': -3, 'sign': 'negative'}: the number is KEPT

Step 1, the step alone:  negative
Step 2, with .assign():  {'number': -3, 'sign': 'negative'}


In [2]:
# Warm-up 2 (no model, no tokens): RunnableBranch is an if / elif / else for chain steps
from langchain_core.runnables import RunnableBranch             # chooses ONE step, by checking conditions in order

# Step 1: the conditions: plain functions that get the WHOLE dict {number, sign} and return True or False
def is_positive(data):
    return data["sign"] == "positive"                           # True only when the sign is "positive"

def is_negative(data):
    return data["sign"] == "negative"                           # True only when the sign is "negative"

# Step 2: one step per branch; the chosen step gets the SAME whole dict
def reply_positive(data):
    return f"{data['number']} is positive: it is more than zero."

def reply_negative(data):
    return f"{data['number']} is negative: it is less than zero."

def reply_zero(data):
    return f"{data['number']} is zero: neither positive nor negative."

# Step 3: RunnableBranch((condition, step), (condition, step), default_step)
number_branch = RunnableBranch(
    (is_positive, RunnableLambda(reply_positive)),              # checked first: if is_positive(...) is True, ONLY reply_positive runs
    (is_negative, RunnableLambda(reply_negative)),              # checked second, only when the first condition was False
    RunnableLambda(reply_zero),                                 # the default (no condition, always last): runs when no condition is True
)

# Step 4: classify and keep (add_sign from Warm-up 1), then branch: {number} → {number, sign} → str
number_chain = add_sign | number_branch
for number in [5, -3, 0]:
    values = {"number": number}                                 # the input dict
    reply = number_chain.invoke(values)                         # each number takes its own branch
    print(reply)

5 is positive: it is more than zero.
-3 is negative: it is less than zero.
0 is zero: neither positive nor negative.


In [3]:
# Step 1: load the API keys, create the model, and create the classifier chain

from typing import Literal                                      # a field that may hold only some fixed values
from pydantic import BaseModel                                  # describes the classifier's answer
from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain_core.runnables import RunnableLambda             # turns a Python function into a chain step
from langchain_core.runnables import RunnablePassthrough        # passes its input through unchanged (with .assign())
from langchain_core.runnables import RunnableBranch             # if / elif / else for chains
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the chat model
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",             # free, on Groq
                        max_retries=5,                          # wait and retry when the free rate limit is hit
                        )

# The classifier's answer: one field that may hold only one of these three words
class Sentiment(BaseModel):
    sentiment: Literal["positive", "negative", "mixed"]

# The classifier prompt, with one {feedback} placeholder
classify_prompt = PromptTemplate.from_template("Classify the sentiment of this customer feedback: {feedback}")

# A model that answers with a Sentiment object instead of an AIMessage
classifier_model = model.with_structured_output(
                        Sentiment,                              # the answer is a Sentiment object
                        method="json_schema",                   # ask Groq for JSON that matches Sentiment
                        )

# A function that keeps only the word inside the Sentiment object
def get_label(result):
    '''Sentiment object → str ("positive", "negative" or "mixed").'''
    label = result.sentiment                                    # read the one field with a dot
    return label

# The classifier chain: {feedback} → PromptValue → Sentiment object → str
classify = classify_prompt | classifier_model | RunnableLambda(get_label)   # RunnableLambda again (Example 4)

label = classify.invoke({"feedback": "Excellent service, my order arrived a day early!"})   # one model call
print(label)                                                    # positive

positive


In [4]:
# Step 2: classify and KEEP: .assign() keeps the input dict and adds the label under the key "sentiment"
# (the same as add_sign in Warm-up 1, with the classifier chain in place of find_sign)
add_sentiment = RunnablePassthrough.assign(sentiment=classify)  # the argument name, sentiment=, becomes the new key

example = {"feedback": "The laptop arrived broken and nobody answers my emails."}
result = add_sentiment.invoke(example)                          # one model call
print(result)                                                   # {feedback} → {feedback, sentiment}: the feedback is still there

{'feedback': 'The laptop arrived broken and nobody answers my emails.', 'sentiment': 'negative'}


In [5]:
# Step 3: one reply chain per sentiment, and RunnableBranch to choose between them

# Three reply chains: each one  {feedback} → PromptValue → AIMessage → str (a reply)
positive_prompt = PromptTemplate.from_template("Thank the customer warmly. Reply in two sentences to this customer feedback: {feedback}")
positive_chain = positive_prompt | model | StrOutputParser()

negative_prompt = PromptTemplate.from_template("Apologise, and say the issue goes to the support team. Reply in two sentences to this customer feedback: {feedback}")
negative_chain = negative_prompt | model | StrOutputParser()

mixed_prompt = PromptTemplate.from_template("Thank the customer for the good points and apologise for the problem. Reply in two sentences to this customer feedback: {feedback}")
mixed_chain = mixed_prompt | model | StrOutputParser()

# The conditions: plain functions that get the WHOLE dict {feedback, sentiment} and return True or False
def is_positive(data):
    return data["sentiment"] == "positive"                      # True only for "positive"

def is_negative(data):
    return data["sentiment"] == "negative"                      # True only for "negative"

# The same shape as number_branch in Warm-up 2
branch = RunnableBranch(
    (is_positive, positive_chain),                              # checked first: if True, ONLY positive_chain runs
    (is_negative, negative_chain),                              # checked second, only when the first condition was False
    mixed_chain,                                                # the default (no condition, always last): runs when no condition is True
)
# The chosen chain gets the SAME dict, so its prompt can still fill {feedback}
# (the extra key "sentiment" is ignored: the reply prompts have no {sentiment} placeholder)

# A test with a hand-made sentiment (no classifier): "mixed" matches no condition, so the default runs
test = {"feedback": "The delivery was late but the product quality is excellent.", "sentiment": "mixed"}
reply = branch.invoke(test)                                     # one model call
print(reply)

Thank you for highlighting the excellent quality of our product—we’re thrilled you’re satisfied with it. We apologize for the delayed delivery and appreciate your patience.


In [6]:
# Step 4: the whole pipeline: {feedback} → {feedback, sentiment} → {feedback, sentiment, reply}
add_reply = RunnablePassthrough.assign(reply=branch)            # run the branch on the dict, ADD its reply under the key "reply"
feedback_chain = add_sentiment | add_reply                      # first classify and keep, then branch and keep

# Three feedbacks: one positive, one negative, one mixed
feedbacks = [
    "Excellent service, my order arrived a day early!",
    "The laptop arrived broken and nobody answers my emails.",
    "The delivery was late but the product quality is excellent.",
]

# One input dict per feedback, for batch()
inputs = []
for feedback in feedbacks:
    inputs.append({"feedback": feedback})

# All three run at the same time; each one takes its own branch (six model calls)
results = feedback_chain.batch(inputs)
for result in results:
    print(f"Feedback:  {result['feedback']}")
    print(f"Sentiment: {result['sentiment']}")
    print(f"Reply:     {result['reply']}")
    print()

Feedback:  Excellent service, my order arrived a day early!
Sentiment: positive
Reply:     Thank you so much for your kind words! We're thrilled your order arrived early and hope you enjoy it.

Feedback:  The laptop arrived broken and nobody answers my emails.
Sentiment: negative
Reply:     We’re very sorry to hear that your laptop arrived damaged and that you haven’t received a response to your emails. I’ve escalated this to our support team, and they will reach out to you shortly to resolve the issue.

Feedback:  The delivery was late but the product quality is excellent.
Sentiment: mixed
Reply:     Thank you for highlighting the excellent quality of our product, and we appreciate your kind words. We apologize for the delayed delivery and are taking steps to improve our shipping times.



# <span style='background :lightgreen' >5. Where Chains Go Next</span>

| Where | What the chain does there | Lecture |
|---|---|---|
| **RAG** | `retriever` → `prompt` → `model` → `parser`: answers questions from your own documents, with `RunnableParallel` and `RunnablePassthrough` | Lec-38 |
| **Agents** | An agent is not a fixed chain: the model decides the next step in a loop. We write that loop ourselves, then use `create_agent()` | Lec-39 to 51 |
| **LangGraph** | Workflows with loops, saved state, branches between agents and human approval | A later lecture |

# <span style='background :lightgreen' >6. Summary</span>

| Without chains (Lec-31) | With chains (LCEL) |
|---|---|
| `prompt.invoke()`, then `model.invoke()`, then `.text`, passed along by our code | `prompt \| model \| StrOutputParser()`, one `invoke()` |
| A loop of calls for many inputs | `chain.batch([...])` |
| Asking for JSON and parsing it ourselves | `PydanticOutputParser` as the last step |
| Reshaping data between steps by hand | `RunnableLambda(function)` |
| Running steps one after the other | `RunnableParallel({...})` |
| An `if` around separate calls | `RunnablePassthrough.assign(...)` + `RunnableBranch(...)` |

### Key takeaways
- Prompts, models, parsers and functions are all **Runnables**: they share `invoke`, `batch` and `stream`, so they can be joined with `|`.
- A **chain** passes each step's output to the next step, and is itself a Runnable.
- **Output parsers** turn the model's answer into a `str` (`StrOutputParser`) or a validated object (`PydanticOutputParser`).
- Three shapes cover most pipelines: **sequential** (`|`, with `RunnableLambda` to reshape data), **parallel** (`RunnableParallel`), and **branching** (`RunnableBranch`, with `.assign()` to keep the input).
- The old `LLMChain`-style classes are deprecated; loops and state belong to LangGraph.
- **Next lecture (Lec-34):** **Document loaders**: getting your own documents into LangChain.

### Check your understanding
1. What do prompt templates, models and output parsers have in common that lets them be joined with `|`?
2. In Example 1b, which step of Example 1a does `StrOutputParser()` replace?
3. What does `translate_chain.batch(inputs)` return, and in which order?
4. What are the two halves of `PydanticOutputParser`'s work, before and after the model? What does `get_format_instructions()` return, and what happens in `parse()` if the model's JSON is wrong?
5. Why could `translate_chain` and `summarize_chain` not be joined directly in Example 4? What did `RunnableLambda` do, and how many arguments may its function take?
6. What does `RunnableParallel` return, and why does that fit `merge_prompt`? In the warm-up of Example 5, why did two steps of 1 second take about 1 second? When does running in parallel save time with a real model, and when not?
7. In Example 6, what would the reply prompt receive without `RunnablePassthrough.assign()`?
8. How does `RunnableBranch` choose a branch, and what happens when no condition is true?
9. Why does LangChain recommend LangGraph instead of chains for workflows with loops?